# Using Moss with AutoGen Agents

This cookbook demonstrates how to use **Moss**, a sub-10ms semantic search engine, as a retrieval tool for **AutoGen** agents. 

By integrating Moss, your agents can access large knowledge bases and retrieve relevant context in milliseconds fast enough to keep multi-agent conversations fluid and responsive.

> **Note:** This cookbook uses the modern **AutoGen v0.4.x** API (via the `autogen-agentchat` package).

## Step 0: Prerequisites
Install the necessary libraries:

In [1]:
%pip install -qU "autogen-agentchat" "autogen-ext[openai]" "inferedge-moss" "python-dotenv"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1: Setup Moss Client
First, initialize the Moss client. We recommend using environment variables (`.env`) for security.

In [4]:
import os
from dotenv import load_dotenv
from inferedge_moss import MossClient, DocumentInfo, QueryOptions

load_dotenv()

# Initialize Moss
moss = MossClient(
    project_id=os.getenv("MOSS_PROJECT_ID"),
    project_key=os.getenv("MOSS_PROJECT_KEY"),
)

## Step 2: Indexing Knowledge Base
Before searching, we need to add some documents to Moss. In this example, we'll index a few customer support FAQs.

In [6]:
async def setup_kb():
    docs = [
        DocumentInfo(
            id="faq-returns",
            text="Our return policy allows items to be returned within 30 days for a full refund.",
            metadata={"category": "returns"}
        ),
        DocumentInfo(
            id="faq-shipping",
            text="Standard shipping takes 5-7 business days. Priority shipping is 2-3 days.",
            metadata={"category": "shipping"}
        )
    ]
    await moss.create_index("customer_support", docs)

# Run once to initialaize
await setup_kb()

## Step 3: Define the Moss Search Tool
Now, we define a tool that the AutoGen agent can call. This function wraps the Moss query. We also include empty result handling, which is a best practice.

In [7]:
async def moss_search(query: str, top_k: int = 3) -> str:
    """Search the customer support knowledge base for relevant policies."""
    options = QueryOptions(top_k=top_k)
    
    # Crucial: Moss queries are asynchronous!
    results = await moss.query("customer_support", query, options)
    
    if not results.docs:
        return "No relevant policies found in the knowledge base."
        
    # Format the results for the LLM
    return "\n\n".join([f"[{doc.score:.3f}] {doc.text}" for doc in results.docs])

## Step 4: Configure the AutoGen Agent
We'll use the modern `AssistantAgent` from `autogen-agentchat`. We register the `moss_search` function as a tool.

In [8]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

# Setup the Model Client
model_client = OpenAIChatCompletionClient(
    model="gpt-4o",
    api_key=os.getenv("OPENAI_API_KEY")
)

# Create the Agent
agent = AssistantAgent(
    name="Customer_Support_Assistant",
    model_client=model_client,
    tools=[moss_search],
    system_message="You are a helpful customer support assistant. Use the 'moss_search' tool to look up policies before answering.",
    reflect_on_tool_use=True,
)

# Run a conversation
async def main():
    await Console(agent.run_stream(task="What is your return policy?"))
    await model_client.close()

await main()

---------- TextMessage (user) ----------
What is your return policy?
---------- ToolCallRequestEvent (Customer_Support_Assistant) ----------
[FunctionCall(id='call_NKSvZvRU8Yjld0NY6ERL9Y9b', arguments='{"query":"return policy"}', name='moss_search')]
---------- ToolCallExecutionEvent (Customer_Support_Assistant) ----------
[FunctionExecutionResult(content='[0.683] Our return policy allows items to be returned within 30 days for a full refund.\n\n[0.036] Standard shipping takes 5-7 business days. Priority shipping is 2-3 days.', name='moss_search', call_id='call_NKSvZvRU8Yjld0NY6ERL9Y9b', is_error=False)]
---------- TextMessage (Customer_Support_Assistant) ----------
Our return policy allows items to be returned within 30 days for a full refund.
